In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_test_sim_S5_20261008_175017.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_test_sim_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
runs = pd.read_csv(RESULT_CSV)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["f_vco", "kind"])
r = runs.iloc[-1]  # pll_test_sim.py writes one row per run; the last one if the CSV holds more
traces = traces[traces["f_vco"] == r.f_vco]
offsets = [c[len("L_vco_"):] for c in runs.columns if c.startswith("L_vco_") and pd.notna(r[c])]
N_DB = 20 * np.log10(r.total_div)  # pad -> VCO
SETTING = (f"c{r.vco_tune_coarse:.0f} min {r.vco_current_min:.0f} max {r.vco_current_max:.0f}, "
           f"C1 {r.set_c1:.0f} C2 {r.set_c2:.0f} Icp {r.set_current:.0f} R1 {r.set_r1:.0f}")
TITLE = f"VCO {r.f_vco / 1e6:g} MHz (ref {r.f_ref / 1e6:g} MHz x 128, pad /{r.total_div:.0f}), {SETTING}"
print(f"{RESULT_CSV.name}: {len(runs)} row(s), showing {r.time}: {'locked' if r.locked == 1 else 'NOT LOCKED'}, "
      f"status {r.status}")

display(pd.Series({
    "setting": SETTING,
    "f_vco set / measured [MHz]": f"{r.f_vco / 1e6:.3f} / {r.f_vco_meas / 1e6:.3f}",
    "lookup table: Vctrl / |Kvco|": f"{r.vctrl_pred:.3f} V / {r.kvco_pred / 1e9:.1f} GHz/V ({r.lut_csv})",
    "pad [MHz] / carrier [dBm]": f"{r.f_out_meas / 1e6:.4f} / {r.carrier_dbm:.1f}",
    "PLL supply": f"{r.pll_vdd:.3f} V, {r.pll_i * 1e3:.3f} mA, {r.pll_p * 1e3:.3f} mW",
    f"RMS jitter {float(offsets[0]):g} Hz - {float(offsets[-1]) / 1e6:g} MHz": (
        f"full curve {r.time_jitter_trace_ps:.2f} ps (noise {r.time_jitter_noise_ps:.2f} ps, "
        f"spurs {r.time_jitter_spur_ps:.2f} ps), markers {r.time_jitter_ps:.2f} ps"),
    "phase jitter at the VCO": (f"full curve {r.phase_jitter_trace_out_mrad * r.total_div / 1e3:.3f} rad, "
                                f"markers {r.phase_jitter_vco_mrad / 1e3:.3f} rad"),
    **{f"L_vco({float(o):g} Hz)": f"{r[f'L_vco_{o}']:.1f} dBc/Hz" for o in offsets},
}, name=r.time).to_frame())

# Spurs on the full curve: power on the pad as measured, and + 20 log10 N (only right if they come from the VCO)
spurs = traces[traces["kind"] == "spur"]
display(pd.DataFrame({"offset [Hz]": spurs["freq"], "on the pad [dBc]": spurs["level_dbm"],
                      "referred to the VCO [dBc]": spurs["level_dbm"] + N_DB})
        .sort_values("on the pad [dBc]", ascending=False).round(1).reset_index(drop=True))

pll_test_sim_S5_20261008_175017.csv: 1 row(s), showing 2026-10-08T17:50:17: locked, status ok


,2026-10-08T17:50:17
setting,"c0 min 10 max 8, C1 3 C2 3 Icp 3 R1 3"
f_vco set / measured [MHz],4096.000 / 4095.997
lookup table: Vctrl / |Kvco|,0.523 V / 34.1 GHz/V (vco_lut_S5_20261001_1616...
pad [MHz] / carrier [dBm],64.0000 / 1.8
PLL supply,"0.750 V, 1.198 mA, 0.898 mW"
RMS jitter 1000 Hz - 10 MHz,"full curve 82.63 ps (noise 80.55 ps, spurs 18...."
phase jitter at the VCO,"full curve 2.126 rad, markers 0.647 rad"
L_vco(1000 Hz),-75.0 dBc/Hz
L_vco(10000 Hz),-73.2 dBc/Hz
L_vco(100000 Hz),-78.8 dBc/Hz


,offset [Hz],on the pad [dBc],referred to the VCO [dBc]
0,1074000.0,-48.3,-12.2
1,3042000.0,-50.0,-13.9
2,963000.0,-54.8,-18.7
3,716400.0,-56.9,-20.8
4,955200.0,-59.9,-23.8
5,10560.0,-72.3,-36.2
6,24600.0,-72.5,-36.3
7,17580.0,-74.0,-37.9
8,22860.0,-74.5,-38.4
9,1758.0,-74.6,-38.5


In [3]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]
OFFSET_AXIS = dict(type="log", tickvals=[1e2, 1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["100 Hz", "1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 300  # points per plotted phase-noise curve (spread over F_start-F_stop); None = every measured point


def trace_of(kind):
    return traces[traces["kind"] == kind]


def offset_axis(f_start, f_stop, f_axis):
    """x-axis settings for an offset plot shown from f_start to f_stop [Hz]; f_axis "log" or "lin"."""
    if f_axis == "log":
        return dict(OFFSET_AXIS, range=[np.log10(f_start), np.log10(f_stop)])
    return dict(type="linear", range=[f_start, f_stop], ticksuffix="Hz", exponentformat="SI")


def in_range(df, f_start, f_stop):
    """Rows of df with the offset (column freq) between f_start and f_stop, so the y axis fits the shown band."""
    return df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)]


def binned(df, f_start, f_stop, f_axis):
    """The curve in df (columns freq, level_dbm) between f_start and f_stop, averaged into N_BINS bins spaced for
    f_axis, in linear power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = in_range(df, f_start, f_stop).sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    if f_axis == "log":
        edges = np.geomspace(f_start, f_stop, N_BINS + 1)
    else:
        edges = np.linspace(f_start, f_stop, N_BINS + 1)
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], edges)
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [4]:
F_start, F_stop = None, None  # offset from the carrier shown [Hz] (may be negative); None = edge of the trace

# Averaged spectrum on the pad around the carrier, as measured (dBm in the trace's RBW, not binned).
d = trace_of("spectrum")
d = d.assign(freq=d["freq"] - d["center"])
d = in_range(d, d["freq"].min() if F_start is None else F_start, d["freq"].max() if F_stop is None else F_stop)
fig = go.Figure()
fig.add_scatter(x=d["freq"], y=d["level_dbm"], mode="lines", line=dict(color=PALETTE[0], width=1.5), showlegend=False,
                hovertemplate="%{x:.4s}Hz: %{y:.1f} dBm<extra></extra>")
fig.update_layout(title=f"Spectrum on the pad, carrier {r.f_out_meas / 1e6:.4f} MHz {r.carrier_dbm:.1f} dBm "
                        f"(RBW {d['rbw'].iloc[0]:g} Hz, {d['averages'].iloc[0]:.0f} averages)<br><sub>{TITLE}</sub>",
                  template="plotly_white", height=500)
fig.update_xaxes(title="offset from the carrier", type="linear", ticksuffix="Hz", exponentformat="SI", gridcolor=GRID)
fig.update_yaxes(title="level [dBm in RBW]", gridcolor=GRID)
save(fig, "spectrum")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_sim_S5_20261008_175017_spectrum.html and .png


In [5]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]
f_axis = "log"  # "log" or "lin"

# Phase noise referred to the VCO (pad + 20 log10 N): the full curve (thick), its noise-only version with the spurs
# replaced by the local floor (dashed), the detected spurs (x at their measured peak, hover: spur power on the pad),
# each span normalized to 1 Hz (thin, no ENBW correction: shows whether the spans line up) and the markers (dots).
# Curves binned to N_BINS points (linear power average, display only).
fig = go.Figure()
for span, d in trace_of("phase_noise").groupby("span"):
    d = d.assign(freq=d["freq"] - d["center"],  # offset from the carrier, normalized to 1 Hz
                 level_dbm=d["level_dbm"] - d["level_dbm"].max() - 10 * np.log10(d["rbw"]))
    d = binned(d[d["freq"] > 10 * d["rbw"]], F_start, F_stop, f_axis)
    fig.add_scatter(x=d["freq"], y=d["level_dbm"] + N_DB, mode="lines", line=dict(color=PALETTE[0], width=0.8),
                    opacity=0.35, showlegend=False,
                    hovertemplate=f"span {span:g} Hz<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
lt_raw = in_range(trace_of("L_trace"), F_start, F_stop)  # spur peaks: from the measured points
lt = binned(trace_of("L_trace"), F_start, F_stop, f_axis)
ln = binned(trace_of("L_noise"), F_start, F_stop, f_axis)
fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + N_DB, mode="lines", line=dict(color=PALETTE[0], width=2.5),
                name="full curve", hovertemplate="full curve: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
fig.add_scatter(x=ln["freq"], y=ln["level_dbm"] + N_DB, mode="lines", line=dict(color=INK, width=1.2, dash="dash"),
                name="noise only", hovertemplate="noise only: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
sp = in_range(trace_of("spur"), F_start, F_stop)
if not sp.empty and not lt_raw.empty:
    fig.add_scatter(x=sp["freq"], y=np.interp(sp["freq"], lt_raw["freq"], lt_raw["level_dbm"]) + N_DB,
                    mode="markers", name="spurs", marker=dict(symbol="x", size=11, color=PALETTE[1]),
                    customdata=sp["level_dbm"],
                    hovertemplate="spur at %{x:.4s}Hz: %{customdata:.1f} dBc on the pad<extra></extra>")
shown = [o for o in offsets if F_start <= float(o) <= F_stop]
fig.add_scatter(x=[float(o) for o in shown], y=[r[f"L_vco_{o}"] for o in shown], mode="markers", name="markers",
                marker=dict(color=PALETTE[2], size=10, line=dict(color="white", width=1)),
                hovertemplate="marker: %{x:.3s}Hz: %{y:.1f} dBc/Hz<extra></extra>")
fig.update_layout(title=f"Phase noise referred to the VCO (pad + {N_DB:.1f} dB)<br><sub>{TITLE}</sub>",
                  template="plotly_white", height=550, legend=dict(orientation="h", y=-0.18))
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **offset_axis(F_start, F_stop, f_axis))
fig.update_yaxes(title="L(f) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "phase_noise")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_sim_S5_20261008_175017_phase_noise.html and .png


In [6]:
# Where the jitter comes from: RMS time jitter integrated from the lowest offset up to f over the measured full curve
# (2 x trapezoid of 10^(L/10), as pll_util.integrate_phase_noise; the end value is the jitter of the table), total and
# noise only. Steps in the total are spurs. Dashed: the markers' value (one point per decade).
fig = go.Figure()
for kind, name, color in (("L_trace", "full curve, total", PALETTE[0]),
                          ("L_noise", "full curve, noise only", PALETTE[2])):
    d = trace_of(kind).sort_values("freq")
    lin = 10 ** (d["level_dbm"].to_numpy() / 10)
    var = np.r_[0.0, np.cumsum((lin[1:] + lin[:-1]) * np.diff(d["freq"].to_numpy()))]  # 2 x trapezoid
    fig.add_scatter(x=d["freq"], y=np.sqrt(var) / (2 * np.pi * r.f_out_meas) * 1e12, mode="lines", name=name,
                    line=dict(color=color, width=2),
                    hovertemplate=f"{name}: %{{y:.2f}} ps up to %{{x:.3s}}Hz<extra></extra>")
fig.add_hline(y=r.time_jitter_ps, line=dict(color=PALETTE[1], width=1.5, dash="dash"),
              annotation_text=f"markers {r.time_jitter_ps:.1f} ps", annotation_position="top left")
fig.update_layout(title=f"Cumulative RMS time jitter (same on the pad and at the VCO)<br><sub>{TITLE}</sub>",
                  template="plotly_white", height=480, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="upper integration limit (offset from the carrier)", gridcolor=GRID, **OFFSET_AXIS)
fig.update_yaxes(title="RMS time jitter [ps]", rangemode="tozero", gridcolor=GRID)
save(fig, "jitter_cumulative")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_test_sim_S5_20261008_175017_jitter_cumulative.html and .png
